# YOLOE-v8-S video: GPU inference, tracking, FPS and playback
1. Runtime > Change runtime type > T4 GPU.
2. Upload `uav_yoloe_video_colab.zip` when prompted, then upload your video (or use Drive).
3. Run all cells. Default: 1280 pixels, FP16, detection every 2 frames plus adaptive refresh and tracking, no skipped source frames.

Speed and accuracy must be measured on your video. Tracking cannot correct persistent classification mistakes. `quality_verified` remains false without labeled evaluation. Use the same clip in `quality` mode to inspect the effect of FP32 and detection every frame.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import files
from pathlib import Path
import zipfile, os, sys
uploaded = files.upload()  # Upload uav_yoloe_video_colab.zip
bundle = next((name for name in uploaded if name.endswith('.zip')), None)
assert bundle, 'Missing project ZIP'
project = Path('/content/uav_yoloe_video')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    for item in archive.infolist():
        assert (project / item.filename).resolve().is_relative_to(project.resolve())
    archive.extractall(project)
os.chdir(project)
sys.path.insert(0, str(project))


In [ ]:
%pip -q install -r lan_chiem/requirements-video.txt -r lan_chiem/requirements-yoloe.txt


In [ ]:
# For large videos, use Drive instead of browser upload.
USE_DRIVE = False
DRIVE_VIDEO = '/content/drive/MyDrive/video.mp4'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    VIDEO = Path(DRIVE_VIDEO)
else:
    uploaded_video = files.upload()
    VIDEO = Path(next(iter(uploaded_video))).resolve()
assert VIDEO.is_file(), VIDEO
import cv2
cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), 'Cannot decode the video'
source_fps = cap.get(cv2.CAP_PROP_FPS)
print('Video:', VIDEO.name, 'FPS:', source_fps,
      'frames:', int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
      'size:', int(cap.get(3)), int(cap.get(4)))
cap.release()


In [ ]:
# Start with 900 frames (~30s at 30 FPS). Set 0 for the full video.
MODE = 'balanced'  # balanced: FP16 + adaptive detection every 2 frames
# MODE = 'quality' # FP32 + detection every frame, for visual comparison
IMAGE_SIZE = 1280
MAX_FRAMES = 900
TARGET_FPS = 15.0  # Reporting target only; this does not guarantee performance.
from datetime import datetime
import yaml
from lan_chiem.yoloe_detector import YOLOEDetector
from lan_chiem.video_predict import VideoOptions, process_video
assert MODE in ('balanced', 'quality')
settings = yaml.safe_load(Path(f'lan_chiem/video_yoloe_{MODE}.yaml').read_text())
settings['target_fps'] = TARGET_FPS
options = VideoOptions(**settings)
detector = YOLOEDetector(weights='yoloe-v8s-seg.pt', device='cuda', imgsz=IMAGE_SIZE)
OUTPUT = project / 'outputs' / (MODE + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
process_video(VIDEO, OUTPUT, detector, options, max_frames=MAX_FRAMES)
print('Saved:', OUTPUT)


In [ ]:
import json
report = json.loads((OUTPUT / 'summary.json').read_text())
fields = ['gpu', 'source_fps', 'processed_frames', 'processed_fps',
          'steady_pipeline_fps_this_run', 'detector_fps_this_run',
          'detector_mean_seconds_this_run', 'ai_frame_fraction_this_run',
          'throughput_target_met', 'completed', 'quality_verified', 'stage_seconds']
print(json.dumps({k: report.get(k) for k in fields}, indent=2))
print('processed_fps: throughput including model loading, warmup and final export.')
print('steady_pipeline_fps_this_run: video processing throughput after loading/warmup.')
print('detector_fps_this_run: detector updates per second spent in detection; not video FPS.')
print('AI fraction: remaining frames use tracking. No source frame skipping.')
print('Target met is a speed result only, not an accuracy result.')


## View the result
Convert MP4V to H.264 for Colab/browser playback. This conversion is separately timed and excluded from pipeline FPS. The output is silent. DETECT/TRACK and processing FPS are overlaid on frames.


In [ ]:
import subprocess, time
from IPython.display import Video, display
playable = OUTPUT / 'preview_h264.mp4'
start = time.perf_counter()
subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
    '-i', str(OUTPUT / 'annotated.mp4'), '-an', '-c:v', 'libx264',
    '-preset', 'fast', '-crf', '20', '-pix_fmt', 'yuv420p',
    '-movflags', '+faststart', str(playable)], check=True)
print('Browser conversion seconds:', round(time.perf_counter()-start, 2))
display(Video(str(playable), embed=True, width=960))


## Inspect quality and compare profiles
Inspect small chairs/tables, bicycle vs motorcycle, signs, missed objects, false detections and track drift. To compare, set MODE to `quality` and rerun the processing, report and playback cells on the same video and MAX_FRAMES. Each run gets a separate directory. Quality mode is a reference configuration, not ground truth. Labeled frames are required to establish precision/recall.


In [ ]:
# Contact sheet of 6 output frames for quick inspection.
from PIL import Image, ImageDraw
capture = cv2.VideoCapture(str(playable))
count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
indices = sorted(set(int(i * max(0, count-1) / 5) for i in range(6)))
canvas = Image.new('RGB', (960, 3*294), 'white')
for k, index in enumerate(indices):
    capture.set(cv2.CAP_PROP_POS_FRAMES, index)
    ok, frame = capture.read()
    if not ok: continue
    thumb = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    thumb.thumbnail((480,270))
    x,y = (k%2)*480,(k//2)*294
    canvas.paste(thumb,(x,y))
    ImageDraw.Draw(canvas).text((x+5,y+272), f'Output frame {index}', fill='black')
capture.release()
canvas.save(OUTPUT / 'contact_sheet.jpg')
display(canvas)


In [ ]:
# Download public outputs, excluding duplicate segment files.
archive_path = Path('/content') / (OUTPUT.name + '_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('preview_h264.mp4', 'summary.json', 'tracks.csv', 'tracks.jsonl', 'contact_sheet.jpg'):
        path = OUTPUT / name
        if path.exists(): archive.write(path, name)
files.download(str(archive_path))
